# Zero-shot real-noise denoising, round 3: DMID's second stage with fewer samples, flips, and a deterministic chain

On real noise, DMID first turns each noisy image into a Gaussian-like one (its released noise-transformed images,
`pre-trained.zip`). It then averages R stochastic samples of a three-step chain started at a fixed step: CC N = 21,
S_t = 2, R = 500; PolyU N = 9, S_t = 3, R = 333; FMDD N = find_N(level), S_t = 3, R = 333; η = 0.85.
Round 2 showed that averaging is worth 0.9 dB on CC over one step.

This round keeps DMID's transformation, step and schedule and changes only the estimator:

* the same stochastic chain with 16 samples, half of them run on the horizontally flipped image (running means at 2–16);
  the same chain without flips (1–8);
* the chain with η = 0, which is deterministic, so a single run has no Monte Carlo error, plus its flip;
* one step at m · σ_D for m < 1, the trend of round 2, with flips.

NT images are matched to noisy inputs (FMDD repeats each ground truth over 4 noise realisations). PolyU 1–50 (dev)
chooses m; PolyU 51–100, CC and FMDD are test. Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import hashlib, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

R_SAMPLES = 16                            # stochastic chain samples per image, alternating original / flipped
ETA = 0.85                                # main_for_real.py: eval_ddpm(eta=0.85)
M_ONE = [0.4, 0.55, 0.7]                  # one step at m · σ_D
DMID_N = {"CC": 21, "PolyU": 9}           # main_for_real.py; FMDD: find_N(2 · noise_level / 255)
DMID_S = {"CC": 2, "PolyU": 3, "FMDD": 3}  # evaluate.sh, the -d settings
DEV_POLYU, SEED, TIME_LIMIT_H = 50, 0, 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
GH = "https://github.com/Li-Tong-621/DMID/releases/download/v1.0/"
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": (GH + "Datasets.zip", 367_018_356),
    "dmid_real": (GH + "3-Real_world.zip", 147_392_996),
    "dmid_nt": (GH + "pre-trained.zip", 352_155_893),
}
with ThreadPoolExecutor(4) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Data: CC (15), PolyU (100), FMDD (48), and DMID's noise-transformed images matched to the noisy inputs

In [ ]:
def read(z, name, mode="RGB"):
    return np.asarray(Image.open(io.BytesIO(z.read(name))).convert(mode))


PAIRS = {  # dataset: (clean folder, noisy folder, clean suffix, noisy suffix, grayscale)
    "CC": ("Datasets/Realworld/CC/GT/", "Datasets/Realworld/CC/Noisy/", "_mean", "_real", False),
    "PolyU": ("Datasets/Realworld/PolyU/PolyU_png/", "Datasets/Realworld/PolyU/Noisy/", "_mean", "_real", False),
    "FMDD": ("Datasets/Realworld/FMDD/gt/", "Datasets/Realworld/FMDD/raw/", "", "", True),
}
DATA = {}
with zipfile.ZipFile(P["datasets"]) as z:
    names = z.namelist()
    for ds, (cdir, ndir, csuf, nsuf, gray) in PAIRS.items():
        cleans = {os.path.splitext(n.split("/")[-1])[0].removesuffix(csuf): n for n in names if n.startswith(cdir) and not n.endswith("/")}
        noisys = {os.path.splitext(n.split("/")[-1])[0].removesuffix(nsuf): n for n in names if n.startswith(ndir) and not n.endswith("/")}
        stems = sorted(set(cleans) & set(noisys), key=natkey)
        mode = "L" if gray else "RGB"
        DATA[ds] = [(s, read(z, cleans[s], mode), read(z, noisys[s], mode)) for s in stems]
SPLIT = {(ds, s): ("dev" if ds == "PolyU" and k < DEV_POLYU else "test") for ds, v in DATA.items() for k, (s, _, _) in enumerate(v)}

# {str(index): {"noise_image": [1, C, H, W] on the 0-255 scale, "noise_level": σ·255}}, indexed in natsorted clean order.
# Each image is matched to the noisy input it was computed from (smallest MSE), which stays unique where FMDD repeats a
# ground truth over 4 noise realisations; if that is not one-to-one, DMID's index order is used instead.
NT, MATCH = {}, {}
with zipfile.ZipFile(P["dmid_nt"]) as z:
    for ds, member in (("CC", "pre-trained/CC.pt"), ("PolyU", "pre-trained/Polyu.pt"), ("FMDD", "pre-trained/FMDD.pt")):
        raw = torch.load(io.BytesIO(z.read(member)), map_location="cpu", weights_only=False)
        noisy = torch.stack([torch.from_numpy(np.ascontiguousarray(n)).float().reshape(-1) for _, _, n in DATA[ds]]).to(DEVICES[0])
        found = []
        for idx, entry in raw.items():
            img = entry["noise_image"].float().reshape(-1, *entry["noise_image"].shape[-2:])  # [C,H,W], 0-255
            flat = (img.mean(0) if ds == "FMDD" else img.permute(1, 2, 0)).reshape(-1).to(DEVICES[0])
            best = ((noisy - flat) ** 2).mean(1).topk(2, largest=False)
            found.append((int(idx), int(best.indices[0]), img / 255, float(entry["noise_level"]), float(best.values[1] / best.values[0])))
        one_to_one = len(found) == len(DATA[ds]) == len({k for _, k, *_ in found})
        for idx, k, img, level, _ in found:
            NT[(ds, DATA[ds][k if one_to_one else idx][0])] = (img, level)
        MATCH[ds] = dict(entries=len(found), one_to_one=one_to_one, same_as_index=sum(i == k for i, k, *_ in found),
                         min_margin=min(m for *_, m in found))
print({ds: len(v) for ds, v in DATA.items()}, MATCH)
assert all(sum(1 for d, _ in NT if d == ds) == len(v) for ds, v in DATA.items()), "missing noise-transformed images"

## 2 · ADM (fp16, SDPA on contiguous q, k, v) and LPIPS on every GPU

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


import lpips

CTX = [dict(dev=d, adm=make_adm(d), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
ADM_STATE.clear()

## 3 · Estimators: one step, and DMID's chain (`utils_sampling.generalized_steps`) at any η

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
ABAR = torch.cumprod(1 - BETAS, 0)
NEG_LOG_ABAR = (-torch.log(ABAR)).numpy()
ABAR32 = torch.cumprod(1 - torch.from_numpy(np.linspace(1e-4, 0.02, 1000)).float(), 0)


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2)."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def sigma_of_step(t):
    ab = float(ABAR[t])
    return 0.5 * math.sqrt((1 - ab) / ab)


def find_n(sigma):
    """utils_cal_N_2.find_N: the N in [2, 500) minimising |sqrt(1 - abar_{N-1}) / (sqrt(abar_{N-1}) sigma) - 1|."""
    ratio = ((1 - ABAR32[1:499]).sqrt() / (ABAR32[1:499].sqrt() * sigma) - 1).abs()
    return int(ratio.argmin()) + 2


@torch.no_grad()
def tweedie(model, y, sigmas):
    """One-step posterior means of y [B,3,H,W] ([0,1] units), embedded at the step matched to sigmas[b]."""
    pairs = [matched(float(s)) for s in sigmas]
    t = torch.tensor([a for a, _ in pairs], dtype=torch.float32, device=y.device)
    ab = torch.tensor([b for _, b in pairs], dtype=torch.float32, device=y.device).view(-1, 1, 1, 1)
    x = ab.sqrt() * (2 * y - 1)
    eps = model(x, t)[:, :3].float()
    return ((x - (1 - ab).sqrt() * eps) / ab.sqrt() + 1) / 2


@torch.no_grad()
def dmid_chain(model, z, n, s, eta, gen):
    """main_for_real.sample_image: x_N = sqrt(abar_{N-1}) (2z - 1), steps seq = range(0, N, N // s)[:-1] + [N - 1],
    DDIM updates with c1 = η sqrt((1 - a_t / a_next)(1 - a_next) / (1 - a_t)); returns the final x0 in [0,1] units, unclamped."""
    seq = list(range(0, n, n // s))[:-1] + [n - 1]
    seq_next = [-1] + seq[:-1]
    x = math.sqrt(float(ABAR32[n - 1])) * (2 * z - 1)
    for i, j in zip(reversed(seq), reversed(seq_next)):
        at, an = float(ABAR32[i]), (float(ABAR32[j]) if j >= 0 else 1.0)
        eps = model(x, torch.full((z.shape[0],), float(i), device=z.device))[:, :3].float()
        x0 = (x - math.sqrt(1 - at) * eps) / math.sqrt(at)
        c1 = eta * math.sqrt((1 - at / an) * (1 - an) / (1 - at))
        c2 = math.sqrt(max(1 - an - c1 ** 2, 0.0))
        x = math.sqrt(an) * x0 + c1 * torch.randn(x0.shape, generator=gen, device=x0.device) + c2 * eps
    return (x + 1) / 2


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(a.shape[0], 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=a.shape[0])[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x, gray):
    """[1,3,H,W] float -> uint8 [C,H,W]; grayscale sets average the three channels, as DMID does for FMDD."""
    x = x[0].clamp(0, 1)
    if gray:
        x = x.mean(0, keepdim=True)
    return (x * 255).round()


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    a3, b3 = (a.expand(3, -1, -1), b.expand(3, -1, -1)) if a.shape[0] == 1 else (a, b)
    with torch.no_grad():
        lpv = lp(a3.float()[None] / 127.5 - 1, b3.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b), lpips=lpv)

## 4 · Every estimator on every image (each family guarded)

In [ ]:
DS_INDEX = {"CC": 1, "PolyU": 2, "FMDD": 3}


def run_image(ctx, job):
    ds, k = job
    stem, clean, _ = DATA[ds][k]
    gray = clean.ndim == 2
    dev, adm = ctx["dev"], ctx["adm"]
    ref = torch.from_numpy(clean[None] if gray else clean.transpose(2, 0, 1).copy()).to(dev).float()
    nt, level = NT[(ds, stem)]
    z = (nt.expand(3, -1, -1) if nt.shape[0] == 1 else nt)[None].to(dev)
    zz = torch.cat([z, z.flip(-1)])
    n = DMID_N.get(ds) or find_n(2 * level / 255)
    s_d = sigma_of_step(n - 1)
    gen = torch.Generator(device=dev).manual_seed(SEED + 100_000 * DS_INDEX[ds] + k)
    outs, errors, info = {}, [], dict(dataset=ds, image=stem, n=n, s=DMID_S[ds], level=level)

    def guard(name, fn):
        try:
            t0 = time.time()
            outs.update(fn())
            info[f"seconds_{name}"] = time.time() - t0
        except Exception:
            errors.append((name, traceback.format_exc(limit=3)))

    def one_step():
        res = {}
        for m in M_ONE:
            o = tweedie(adm, zz, [m * s_d] * 2)
            res[f"one step, m={m:g}"] = o[:1]
            res[f"one step ×2 (flip), m={m:g}"] = (o[:1] + o[1:].flip(-1)) / 2
        return res

    def deterministic():
        o = dmid_chain(adm, zz, n, DMID_S[ds], 0.0, gen)
        return {"chain η=0": o[:1], "chain η=0 ×2 (flip)": (o[:1] + o[1:].flip(-1)) / 2}

    def stochastic():
        res, total, orig = {}, 0.0, 0.0
        for b in range(R_SAMPLES // 2):
            o = dmid_chain(adm, zz, n, DMID_S[ds], ETA, gen)
            total, orig = total + o[:1] + o[1:].flip(-1), orig + o[:1]
            if b + 1 in (1, 2, 4, 8):
                res[f"chain η={ETA:g}, {b + 1} samples"] = orig / (b + 1)
            if 2 * (b + 1) in (2, 4, 8, 16):
                res[f"chain η={ETA:g}, {2 * (b + 1)} samples, half flipped"] = total / (2 * (b + 1))
        return res

    guard("one_step", one_step)
    guard("deterministic", deterministic)
    guard("stochastic", stochastic)
    rows = [dict(dataset=ds, image=stem, split=SPLIT[(ds, stem)], method=m, **scores(to_u8(o, gray), ref, ctx["lpips"])) for m, o in outs.items()]
    return rows, info, errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [(ds, k) for ds in ("CC", "FMDD", "PolyU") for k in range(len(DATA[ds]))]
res, fail = worker_pool(jobs[:2], run_image)   # smoke test before the full run
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e for _, _, errs in res for e in errs][:2],
      [{k: round(v, 1) for k, v in info.items() if k.startswith("seconds")} for _, info, _ in res])
res2, fail2 = worker_pool(jobs[2:], run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
INFO = pd.DataFrame([info for _, info, _ in res])
METHOD_ERRORS = [(info["dataset"], info["image"], name, tb) for _, info, errs in res for name, tb in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
INFO.to_csv(f"{OUT}/image_info.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(jobs)} images, {len(fail)} failed images, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)
print(INFO.groupby("dataset")[["n", "level"] + [c for c in INFO if c.startswith("seconds")]].mean().round(2).to_string())

## 5 · DMID's released outputs, rescored with the same code

In [ ]:
key = lambda a: hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()


def as_tensor(a):
    """uint8 [H,W] or [H,W,3] -> float [C,H,W] on the first GPU."""
    return torch.from_numpy(a[None] if a.ndim == 2 else a.transpose(2, 0, 1).copy()).float().to(DEVICES[0])


CLEAN_STEMS = {}
for ds, v in DATA.items():
    for s, c, _ in v:
        CLEAN_STEMS.setdefault((ds, key(c)), []).append(s)
dmid_rows = []
with zipfile.ZipFile(P["dmid_real"]) as z:
    for n in z.namelist():
        m = re.match(r"3-Real_world/(CC|PolyU|FMDD)-([dp])/denoise/(\d+)_denoise\.png$", n)
        if not m:
            continue
        ds, variant, idx = m.groups()
        mode = "L" if ds == "FMDD" else "RGB"
        clean, den = read(z, f"3-Real_world/{ds}-{variant}/clean/{idx}_clean.png", mode), read(z, n, mode)
        stems = CLEAN_STEMS.get((ds, key(clean)), [])
        # DMID numbers its outputs from 0 in natsorted clean order, which picks among repeated ground truths (FMDD)
        by_index = DATA[ds][int(idx)][0] if int(idx) < len(DATA[ds]) else None
        stem = by_index if by_index in stems else (stems[0] if stems else f"unmatched {idx}")
        dmid_rows.append(dict(dataset=ds, image=stem, split=SPLIT.get((ds, stem), "test"),
                              method=f"DMID-{variant} (released)", **scores(as_tensor(den), as_tensor(clean), CTX[0]["lpips"])))
DMID = pd.DataFrame(dmid_rows)
print("DMID outputs:", DMID.groupby(["method", "dataset"]).size().to_dict(), "unmatched:", int(DMID.image.str.startswith("unmatched").sum()))

## 6 · Results

In [ ]:
ALL = pd.concat([SCORES, DMID], ignore_index=True)
ALL.to_csv(f"{OUT}/all_scores.csv", index=False)
mean = ALL.groupby(["method", "dataset"])[["psnr", "ssim", "lpips"]].mean()
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} (all images)")
    print(mean[metric].unstack().round(4 if metric != "psnr" else 3).to_string())
dev = SCORES[SCORES.split == "dev"].groupby("method").psnr.mean()
chosen = {f: dev[[m for m in dev.index if m.startswith(f)]].idxmax() for f in ("one step, m=", "one step ×2 (flip), m=")}
print("\nchosen on PolyU 1-50 (dev):", chosen)
wide = ALL.pivot_table(index=["dataset", "image"], columns="method", values="psnr")
for m in [*chosen.values(), "chain η=0 ×2 (flip)", f"chain η={ETA:g}, 8 samples", f"chain η={ETA:g}, 16 samples, half flipped"]:
    if m in wide:
        d = (wide[m] - wide["DMID-d (released)"]).dropna()
        print(f"\n{m} − DMID-d (released), PSNR: " + ", ".join(f"{ds} {g.mean():+.3f} ({int((g > 0).sum())}/{g.size})" for ds, g in d.groupby(level=0)))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, chosen_on_dev=chosen, match=MATCH, eta=ETA, samples=R_SAMPLES, m_one=M_ONE,
                   failed_images=[j for j, _ in fail], failed_methods=[e[:3] for e in METHOD_ERRORS]), f, indent=1)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")